In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn

In [ ]:
df = pd.read_csv('/content/predictive_maintenance.csv')
display(df.head())

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (8000, 8)
X_test shape: (2000, 8)
y_train shape: (8000,)
y_test shape: (2000,)


In [ ]:
df.isnull().sum()

,0
UDI,0
Product ID,0
Type,0
Air temperature [K],0
Process temperature [K],0
Rotational speed [rpm],0
Torque [Nm],0
Tool wear [min],0
Target,0
Failure Type,0


In [ ]:
df = df.drop('UDI', axis=1)
display(df.head())

,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type
0,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure
1,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure
2,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure
3,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure
4,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure


In [ ]:
df['Failure_Type_Encoded'] = pd.factorize(df['Failure Type'])[0]
display(df[['Failure Type', 'Failure_Type_Encoded']].head())
print(df['Failure_Type_Encoded'].unique())

,Failure Type,Failure_Type_Encoded
0,No Failure,0
1,No Failure,0
2,No Failure,0
3,No Failure,0
4,No Failure,0


[0 1 2 3 4 5]


In [ ]:
unique_failure_types = df['Failure Type'].unique()
failure_type_mapping = {'No Failure': 0}
current_code = 1
for ft in unique_failure_types:
    if ft != 'No Failure':
        failure_type_mapping[ft] = current_code
        current_code += 1

df['Failure_Type_Encoded'] = df['Failure Type'].map(failure_type_mapping)
display(df[['Failure Type', 'Failure_Type_Encoded']].head())
print(df['Failure_Type_Encoded'].unique())

,Failure Type,Failure_Type_Encoded
0,No Failure,0
1,No Failure,0
2,No Failure,0
3,No Failure,0
4,No Failure,0


[0 1 2 3 4 5]


In [ ]:
# Define y as the 'Target' column
y = df['Target']

# Create X by dropping the original 'Target' and 'Failure Type' columns.
# The 'Failure_Type_Encoded' column will be retained as a feature.
X = df.drop(columns=['Target', 'Failure Type', 'UDI'], errors='ignore')

# Label encode 'Product ID' to numbers
X['Product ID_Encoded'] = pd.factorize(X['Product ID'])[0]
# Drop the original 'Product ID' column from X
X = X.drop(columns=['Product ID'])

# Label encode 'Type' to numbers
X['Type_Encoded'] = pd.factorize(X['Type'])[0]
# Drop the original 'Type' column from X
X = X.drop(columns=['Type'])

print("Updated Features (X) head:")
display(X.head())
print("\nData types of X:")
print(X.dtypes)

Updated Features (X) head:


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Failure_Type_Encoded,Product ID_Encoded,Type_Encoded
0,298.1,308.6,1551,42.8,0,0,0,0
1,298.2,308.7,1408,46.3,3,0,1,1
2,298.1,308.5,1498,49.4,5,0,2,1
3,298.2,308.6,1433,39.5,7,0,3,1
4,298.2,308.7,1408,40.0,9,0,4,1



Data types of X:
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Failure_Type_Encoded         int64
Product ID_Encoded           int64
Type_Encoded                 int64
dtype: object


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Initialize the Logistic Regression model
log_reg_model = LogisticRegression(random_state=42, solver='liblinear')

# Train the model on the training data
log_reg_model.fit(X_train, y_train)

# Make predictions on the test data
y_pred = log_reg_model.predict(X_test)

# Calculate the accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression Model Accuracy: {accuracy:.4f}")

Logistic Regression Model Accuracy: 0.9940


In [ ]:
from sklearn.metrics import classification_report

# Generate a classification report
class_report = classification_report(y_test, y_pred)

print("Classification Report:")
print(class_report)

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1939
           1       0.90      0.90      0.90        61

    accuracy                           0.99      2000
   macro avg       0.95      0.95      0.95      2000
weighted avg       0.99      0.99      0.99      2000

